# 06 - Preparación del dashboard

**Proyecto:** Analítica de ventas de videojuegos en Databricks  
**Autor:** Darren J. Blackman M.  
**Asignatura:** Análisis de Datos y Toma de Decisiones en Computación

## Objetivo
Crear tablas resumidas para el dashboard de Databricks. El tablero debe contener entre 3 y 5 gráficos, 1 o 2 filtros, 2 a 4 indicadores y una interpretación textual.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]
ESQUEMA = "default"

def tabla(nombre):
    return f"`{CATALOGO}`.`{ESQUEMA}`.`{nombre}`"

print(f"Catálogo activo: {CATALOGO}")
print(f"Esquema utilizado: {ESQUEMA}")

Catálogo activo: workspace
Esquema utilizado: default


In [0]:
TABLA_LIMPIA = tabla("vgsales_limpio")
TABLA_CLUSTERS = tabla("vgsales_clusters")

df = spark.table(TABLA_LIMPIA)
df.createOrReplaceTempView("vgsales_limpio_sql")

try:
    df_clusters = spark.table(TABLA_CLUSTERS)
    df_clusters.createOrReplaceTempView("vgsales_clusters_sql")
    HAY_CLUSTERS = True
except Exception:
    HAY_CLUSTERS = False
    print("Aviso: ejecute primero 05_analitica_avanzada para incluir el gráfico de clústeres.")

In [0]:
# KPI: total de registros y ventas
total_juegos = df.count()
ventas_globales = float(df.agg(F.sum("Global_Sales")).first()[0])

plataforma_lider = (
    df.groupBy("Platform").agg(F.sum("Global_Sales").alias("Ventas"))
      .orderBy(F.desc("Ventas")).first()
)
genero_lider = (
    df.groupBy("Genre").agg(F.sum("Global_Sales").alias("Ventas"))
      .orderBy(F.desc("Ventas")).first()
)

kpis = spark.createDataFrame([(
    total_juegos,
    round(ventas_globales, 2),
    plataforma_lider["Platform"],
    round(float(plataforma_lider["Ventas"]), 2),
    genero_lider["Genre"],
    round(float(genero_lider["Ventas"]), 2),
)], [
    "Total_Videojuegos", "Ventas_Globales", "Plataforma_Lider",
    "Ventas_Plataforma_Lider", "Genero_Lider", "Ventas_Genero_Lider"
])

kpis.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_kpis"))
display(kpis)

Total_Videojuegos,Ventas_Globales,Plataforma_Lider,Ventas_Plataforma_Lider,Genero_Lider,Ventas_Genero_Lider
16327,8820.36,PS2,1233.46,Action,1722.88


In [0]:
# Top 10 plataformas
plataformas = (
    df.groupBy("Platform")
      .agg(
          F.count("*").alias("Cantidad_Juegos"),
          F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales")
      )
      .orderBy(F.desc("Ventas_Globales"))
      .limit(10)
)
plataformas.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_plataformas"))

In [0]:
# Ventas por género
generos = (
    df.groupBy("Genre")
      .agg(
          F.count("*").alias("Cantidad_Juegos"),
          F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales")
      )
      .orderBy(F.desc("Ventas_Globales"))
)
generos.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_generos"))

In [0]:
# Evolución anual
anios = (
    df.groupBy("Year")
      .agg(F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales"))
      .orderBy("Year")
)
anios.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_anios"))

In [0]:
# Ventas regionales
regiones = spark.createDataFrame([
    ("Norteamérica", float(df.agg(F.sum("NA_Sales")).first()[0])),
    ("Europa", float(df.agg(F.sum("EU_Sales")).first()[0])),
    ("Japón", float(df.agg(F.sum("JP_Sales")).first()[0])),
    ("Otras regiones", float(df.agg(F.sum("Other_Sales")).first()[0])),
], ["Region", "Ventas"])
regiones.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_regiones"))

In [0]:
# Top 10 videojuegos
top_juegos = (
    df.select("Name", "Platform", "Genre", "Year", "Global_Sales")
      .orderBy(F.desc("Global_Sales"))
      .limit(10)
)
top_juegos.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_top_juegos"))

In [0]:
# Resumen de clústeres (si el notebook 05 ya se ejecutó)
if HAY_CLUSTERS:
    clusters = (
        df_clusters.groupBy("Cluster", "Etiqueta_Cluster")
        .agg(
            F.count("*").alias("Cantidad_Juegos"),
            F.round(F.avg("Global_Sales"), 3).alias("Promedio_Global"),
            F.round(F.sum("Global_Sales"), 2).alias("Ventas_Globales")
        )
        .orderBy("Promedio_Global")
    )
    clusters.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_clusters"))
    display(clusters)

Cluster,Etiqueta_Cluster,Cantidad_Juegos,Promedio_Global,Ventas_Globales
0,Ventas bajas,15343,0.303,4653.8
1,Ventas altas,984,4.234,4166.56


In [0]:
# Texto interpretativo para el dashboard
anio_lider = (
    df.groupBy("Year").agg(F.sum("Global_Sales").alias("Ventas"))
      .orderBy(F.desc("Ventas")).first()
)
region_lider = regiones.orderBy(F.desc("Ventas")).first()

texto = (
    f"La plataforma líder es {plataforma_lider['Platform']} con "
    f"{float(plataforma_lider['Ventas']):,.2f} millones de unidades. "
    f"El género con mayor venta acumulada es {genero_lider['Genre']}. "
    f"El año de mayor venta es {int(anio_lider['Year'])}, mientras que "
    f"{region_lider['Region']} concentra el mayor volumen regional."
)
interpretacion = spark.createDataFrame([(texto,)], ["Interpretacion"])
interpretacion.write.format("delta").mode("overwrite").saveAsTable(tabla("dashboard_interpretacion"))
display(interpretacion)

Interpretacion
"La plataforma líder es PS2 con 1,233.46 millones de unidades. El género con mayor venta acumulada es Action. El año de mayor venta es 2008, mientras que Norteamérica concentra el mayor volumen regional."


## Diseño recomendado del dashboard

### Indicadores (4)
1. Total de videojuegos.  
2. Ventas globales.  
3. Plataforma líder.  
4. Género líder.

### Gráficos (5)
1. Barras: Top 10 plataformas.  
2. Barras o pastel: ventas por género.  
3. Línea: evolución anual.  
4. Barras: ventas por región.  
5. Barras: cantidad de juegos por clúster.

### Filtros (2)
1. Plataforma.  
2. Rango de años.

### Texto
Agregue la tabla `dashboard_interpretacion` como texto o copie su contenido en un bloque de texto del dashboard.

> **PENDIENTE DEL ESTUDIANTE:** crear el AI/BI Dashboard desde la interfaz, publicar/compartirlo según las opciones disponibles en su cuenta y guardar el enlace y las capturas.

## Evidencias que debe capturar
1. Listado de tablas `dashboard_*` en el catálogo.  
2. Dashboard completo.  
3. Dashboard con filtro de plataforma.  
4. Dashboard con rango de años.  
5. Vista del texto interpretativo.